# 예제 03. 요약 압축과 정보 보존율 측정 · Example 03 — Summary Compression and Measuring Information Retention

전산실험 2 · AI Agent 만들기 · **4주차** · 예상 소요 45분
Computational Experiments 2 · Building AI Agents · **Week 4** · about 45 minutes

## 목표 · Objectives
- 오래된 대화를 요약으로 갈아끼운다
- `drop` 과 `summary` 의 **정보 보존율을 측정**해 비교한다
- 요약에서 특히 잘 사라지는 것이 무엇인지 확인한다
- 요약 프롬프트를 개선한다

> **EN**
> - Replace the old part of a conversation with a summary
> - **Measure and compare the information retention** of `drop` versus `summary`
> - Find out what a summary is especially prone to losing
> - Improve the summarisation prompt

> **probe 방식 평가**를 처음 씁니다.
> 압축 품질을 눈으로 읽어 판단하는 대신, **답할 수 있는지**로 판정합니다.
>
> **EN — our first use of probe-based evaluation.** Instead of judging compression
> quality by reading it, we judge it by **whether the question can still be
> answered.**

### 준비 — 공용 모듈과 데이터 만들기 / Setup — write the shared modules and data

이 수업 저장소에는 노트북(`.ipynb`)만 둡니다. 그래서 실습에 필요한 공용 모듈과
데이터를 **아래 셀들이 직접 파일로 만들어 냅니다.** 내려받지 않으므로 네트워크가
느려도, 저장소가 비공개로 바뀌어도 그대로 동작합니다.

아래 셀을 **위에서부터 차례로 한 번씩** 실행한 뒤 다음으로 넘어가세요.
내용을 지금 읽을 필요는 없습니다. 만들어진 `.py` 파일은 왼쪽 파일 탐색기에서
언제든 열어 볼 수 있습니다.

The course repository contains notebooks only. The cells below **write the shared
modules and data files themselves** instead of downloading them. Run them once,
top to bottom, then continue. You do not need to read them now — the generated
`.py` files can be opened from the file browser on the left.

In [ ]:
%%writefile kmu_llm.py
"""전산실험2 - AI Agent 만들기 : 공용 LLM 래퍼.

계명대 통계학과 서버와 Google Gemini를 같은 방식으로 호출한다.
둘 다 OpenAI 호환 API를 제공하므로 openai SDK 하나로 통일하고
base_url 과 model 만 바꾼다.

EN — Shared LLM wrapper for "Computational Experiments 2: Building AI Agents".
Calls the Keimyung Statistics department server and Google Gemini the same way.
Both expose an OpenAI-compatible API, so we use the single openai SDK and only
swap `base_url` and `model`.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 이 저장소는 공개(public)이므로 **학과 서버 주소와 API 키를 코드에 적지
 않는다.** 둘 다 수업시간에 공개하고, 각자 Colab Secrets 에 저장해서 쓴다.

     KMU_BASE_URL  = 수업시간에 공개  (예: https://.../v1)
     KMU_API_KEY   = 수업시간에 공개

 왼쪽 열쇠 아이콘(🔑) → 새 secret → 노트북 액세스 토글 켜기

 EN — This repository is PUBLIC, so never write the department server URL or
 the API key in code. Both are announced in class; store them in Colab Secrets.
     Key icon (🔑) on the left → new secret → turn on notebook access
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

사용 예 · Example:
    from kmu_llm import LLM, ask, check_server

    check_server()                       # 서버가 살아 있는지 먼저 확인
                                         # Check the server is alive first
    print(ask("표준편차를 한 문장으로 설명해줘"))

    llm = LLM(provider="gemini")         # 백업 경로 / Fallback route
    print(llm.chat([{"role": "user", "content": "안녕"}]))
"""

from __future__ import annotations

import os
import time
from typing import Any, Iterator

__all__ = [
    "KMU_MODEL",
    "GEMINI_BASE_URL",
    "GEMINI_MODEL",
    "PROVIDERS",
    "LLM",
    "ask",
    "get_secret",
    "get_key",
    "get_base_url",
    "check_server",
]

# --------------------------------------------------------------------------
# 서버 설정 / Server configuration
# --------------------------------------------------------------------------
# 학과 서버 주소는 여기에 적지 않는다. Colab Secrets 의 KMU_BASE_URL 에서 읽는다.
# 모델 태그는 공개해도 무방하므로 상수로 둔다.
# EN — The department server URL is never written here; it is read from the
#      Colab secret KMU_BASE_URL. The model tag is harmless to publish, so it
#      stays a constant.
KMU_MODEL = os.environ.get("KMU_MODEL", "gemma4")

# 백업 경로. Google AI Studio 에서 무료 키를 발급받아 쓴다. (공개 주소)
# EN — Fallback route. Get a free key from Google AI Studio. (public URL)
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GEMINI_MODEL = "gemini-2.0-flash"

PROVIDERS: dict[str, dict[str, Any]] = {
    "kmu": {
        "base_url": None,  # 실행 시점에 Secrets 에서 읽는다 / read from Secrets at runtime
        "base_url_key": "KMU_BASE_URL",
        "model": KMU_MODEL,
        "key_name": "KMU_API_KEY",
        "label": "계명대 학과 서버",
    },
    "gemini": {
        "base_url": GEMINI_BASE_URL,
        "base_url_key": None,
        "model": GEMINI_MODEL,
        "key_name": "GEMINI_API_KEY",
        "label": "Google Gemini (백업)",
    },
}


# --------------------------------------------------------------------------
# 비밀정보 가져오기 / Loading secrets
# --------------------------------------------------------------------------
def get_secret(name: str, *, prompt: bool = True, secret: bool = True) -> str:
    """설정값을 안전한 순서로 찾는다.

    1) Colab Secrets (왼쪽 열쇠 아이콘)  2) 환경변수  3) 직접 입력

    값을 코드에 적지 않는 것이 핵심이다. 노트북이나 모듈을 GitHub 에 올리는
    순간 코드에 적힌 값은 전 세계에 공개된다. API 키뿐 아니라 **내부 서버
    주소도** 마찬가지다.

    EN — Look up a configuration value in a safe order:
    1) Colab Secrets (key icon on the left)  2) environment variable  3) prompt.
    The point is to keep the value out of the source. The moment a notebook or
    module is pushed to GitHub, anything written in the code is world-readable —
    the internal server URL just as much as the API key.

    Parameters
    ----------
    secret : True 면 입력을 화면에 표시하지 않는다(getpass).
             주소처럼 오타 확인이 중요한 값은 False 로 준다.
             EN — True hides the typed value (getpass). Pass False for values
             like a URL where you need to see typos.
    """
    # 1) Colab Secrets
    try:
        from google.colab import userdata  # type: ignore

        value = userdata.get(name)
        if value:
            return value.strip()
    except Exception:
        pass

    # 2) 환경변수 / environment variable
    value = os.environ.get(name)
    if value:
        return value.strip()

    # 3) 직접 입력 / type it in
    if not prompt:
        raise RuntimeError(
            f"'{name}' 를 찾을 수 없습니다.\n"
            f"  Colab 왼쪽 열쇠 아이콘(🔑) → 새 secret → 이름 '{name}'\n"
            f"  값은 수업시간에 공개합니다. 노트북 액세스 토글도 켜세요."
        )

    print(f"[{name}] 를 찾지 못했습니다. 직접 입력해 주세요.")
    print(f"   (Colab Secrets 에 '{name}' 로 저장해 두면 다시 묻지 않습니다)")

    if secret:
        import getpass

        value = getpass.getpass(f"{name}: ").strip()
    else:
        value = input(f"{name}: ").strip()

    if not value:
        raise RuntimeError(f"'{name}' 에 빈 값이 입력되었습니다.")
    os.environ[name] = value  # 같은 런타임 안에서는 다시 묻지 않는다 / ask only once per runtime
    return value


def get_key(key_name: str = "KMU_API_KEY", *, prompt: bool = True) -> str:
    """API 키를 가져온다. (get_secret 의 별칭 — 1주차 이름 유지)

    EN — Fetch the API key. Alias of get_secret, kept under the week-1 name.
    """
    return get_secret(key_name, prompt=prompt, secret=True)


def get_base_url(provider: str = "kmu", *, prompt: bool = True) -> str:
    """서버 주소를 가져온다. 끝의 '/' 는 떼고 돌려준다.

    학과 서버 주소는 저장소에 적지 않으므로 Secrets 에서 읽는다.
    '/chat/completions' 까지 붙여서 저장한 경우도 알아서 잘라낸다.

    EN — Return the server URL with any trailing '/' removed. The department
    URL is never stored in the repository, so it comes from Secrets. If you
    saved the full '/chat/completions' path, it is trimmed for you.
    """
    conf = PROVIDERS[provider]

    if conf["base_url"]:
        return str(conf["base_url"]).rstrip("/")

    url = get_secret(conf["base_url_key"], prompt=prompt, secret=False).rstrip("/")

    # openai SDK 는 base_url 뒤에 /chat/completions 를 스스로 붙인다.
    # 주소를 통째로 저장한 경우를 대비해 정리해 준다.
    # EN — The openai SDK appends /chat/completions itself, so strip it here in
    #      case the whole path was saved as the secret.
    for suffix in ("/chat/completions", "/completions"):
        if url.endswith(suffix):
            url = url[: -len(suffix)]

    conf["base_url"] = url  # 같은 세션에서는 재사용 / reuse within the session
    return url


# --------------------------------------------------------------------------
# LLM 클라이언트 / LLM client
# --------------------------------------------------------------------------
class LLM:
    """OpenAI 호환 서버를 감싼 얇은 클라이언트.

    EN — A thin client wrapping any OpenAI-compatible server.

    Parameters
    ----------
    provider : "kmu" | "gemini"
        어느 서버를 쓸지. 기본은 학과 서버.
        EN — Which server to use. Defaults to the department server.
    model, base_url, api_key :
        직접 지정하면 기본값을 덮어쓴다. (보통 지정하지 않는다)
        EN — Override the defaults. You normally leave these alone.
    """

    def __init__(
        self,
        provider: str = "kmu",
        model: str | None = None,
        base_url: str | None = None,
        api_key: str | None = None,
        timeout: float = 120.0,
    ) -> None:
        if provider not in PROVIDERS:
            raise ValueError(
                f"알 수 없는 provider: {provider!r}. 가능한 값: {list(PROVIDERS)}"
            )
        conf = PROVIDERS[provider]

        self.provider = provider
        self.label = conf["label"]
        self.model = model or conf["model"]
        self.base_url = base_url or get_base_url(provider)
        self.api_key = api_key or get_key(conf["key_name"])

        try:
            from openai import OpenAI
        except ImportError as exc:  # pragma: no cover
            raise ImportError(
                "openai 패키지가 필요합니다.  !pip install -q openai"
            ) from exc

        self.client = OpenAI(
            api_key=self.api_key, base_url=self.base_url, timeout=timeout
        )

    def __repr__(self) -> str:
        # 주소는 출력하지 않는다. 화면 공유·스크린샷으로 새어나가기 쉽다.
        # EN — Never print the URL; screen shares and screenshots leak it easily.
        return f"LLM(provider={self.provider!r}, model={self.model!r})"

    # -- 기본 호출 / Core calls -------------------------------------------
    def raw(self, messages: list[dict[str, str]], **kw: Any):
        """응답 객체 전체를 그대로 돌려준다. usage(토큰 수) 확인용.

        EN — Return the full response object, so you can inspect `usage`
        (token counts).
        """
        params: dict[str, Any] = {
            "model": self.model,
            "messages": messages,
            "temperature": 0.7,
            "max_tokens": 1024,
        }
        params.update(kw)
        return self.client.chat.completions.create(**params)

    def chat(self, messages: list[dict[str, str]], **kw: Any) -> str:
        """대화 메시지를 보내고 응답 '텍스트'만 돌려준다.

        EN — Send the message list and return only the response text.
        """
        response = self.raw(messages, **kw)
        content = response.choices[0].message.content
        return (content or "").strip()

    def stream(self, messages: list[dict[str, str]], **kw: Any) -> Iterator[str]:
        """응답을 조각(chunk) 단위로 흘려보낸다. 타이핑되는 효과.

        EN — Yield the response chunk by chunk, giving the typing effect.
        """
        kw["stream"] = True
        for chunk in self.raw(messages, **kw):
            if not chunk.choices:
                continue
            piece = chunk.choices[0].delta.content
            if piece:
                yield piece

    # -- 편의 메서드 / Convenience methods ---------------------------------
    def ask(self, prompt: str, system: str | None = None, **kw: Any) -> str:
        """한 번짜리 질문. messages 를 매번 만들기 번거로울 때 쓴다.

        EN — A one-shot question, for when building `messages` by hand is
        tedious.
        """
        messages: list[dict[str, str]] = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
        return self.chat(messages, **kw)

    def ask_retry(
        self,
        prompt: str,
        system: str | None = None,
        *,
        n_retry: int = 3,
        wait: float = 2.0,
        **kw: Any,
    ) -> str:
        """일시적인 네트워크·서버 오류를 재시도한다.

        30회 반복 실험처럼 호출이 많은 실습에서 한 번의 실패로 전체가
        멈추지 않도록 하기 위한 것이다.

        EN — Retry transient network/server errors, so that a lab with many
        calls (a 30-repetition experiment, say) does not die on one failure.
        """
        last_error: Exception | None = None
        for attempt in range(1, n_retry + 1):
            try:
                return self.ask(prompt, system, **kw)
            except Exception as exc:  # 네트워크·서버 오류 전반 / any network or server error
                last_error = exc
                if attempt < n_retry:
                    print(f"   [재시도 {attempt}/{n_retry - 1}] {type(exc).__name__}")
                    time.sleep(wait * attempt)  # 점점 더 오래 기다린다 / back off progressively
        raise RuntimeError(f"{n_retry}회 시도 모두 실패했습니다.") from last_error

    def count_tokens(self, messages: list[dict[str, str]]) -> int:
        """입력 토큰 수를 서버에 물어본다. 출력은 1토큰으로 잘라 낭비를 줄인다.

        EN — Ask the server how many input tokens the messages use; the output
        is capped at 1 token to avoid waste.
        """
        return self.raw(messages, max_tokens=1).usage.prompt_tokens


# --------------------------------------------------------------------------
# 모듈 수준 편의 함수 / Module-level convenience functions
# --------------------------------------------------------------------------
_default_llm: LLM | None = None


def ask(
    prompt: str, system: str | None = None, *, provider: str = "kmu", **kw: Any
) -> str:
    """가장 간단한 한 줄 호출.  ask("안녕") -> "안녕하세요..."

    내부적으로 LLM 객체를 한 번만 만들어 재사용한다.

    EN — The simplest one-line call. Internally it builds one LLM object and
    reuses it.
    """
    global _default_llm
    if _default_llm is None or _default_llm.provider != provider:
        _default_llm = LLM(provider=provider)
    return _default_llm.ask(prompt, system, **kw)


# --------------------------------------------------------------------------
# 서버 진단 / Server diagnostics
# --------------------------------------------------------------------------
def check_server(provider: str = "kmu", base_url: str | None = None) -> dict[str, Any]:
    """서버가 살아 있는지, 어떤 모델이 올라가 있는지 확인한다.

    수업 첫 실습. 실패하면 무엇을 확인해야 하는지 함께 출력한다.

    EN — Check that the server is alive and see which models it serves. This is
    the very first lab of the course; on failure it prints what to check.

    Returns
    -------
    dict : {"ok": bool, "models": list[str], "error": str | None}
    """
    import requests

    conf = PROVIDERS[provider]
    base_url = (base_url or get_base_url(provider)).rstrip("/")
    api_key = get_key(conf["key_name"])

    # 주소 전체를 출력하지 않는다. 호스트만 가려서 보여준다.
    # EN — Never print the full URL; show the host masked.
    print(f"조회 중: GET {_mask_url(base_url)}/models")

    try:
        response = requests.get(
            f"{base_url}/models",
            headers={"Authorization": f"Bearer {api_key}"},
            timeout=15,
        )
    except Exception as exc:
        print(f"\n[실패] 서버에 연결하지 못했습니다: {type(exc).__name__}")
        _print_troubleshooting()
        return {"ok": False, "models": [], "error": str(exc)}

    if response.status_code != 200:
        print(f"\n[실패] HTTP {response.status_code}")
        print(f"   응답: {response.text[:300]}")
        if response.status_code in (401, 403):
            print("   -> API 키가 잘못되었거나 만료되었을 가능성이 높습니다.")
        elif response.status_code == 404:
            print("   -> KMU_BASE_URL 이 '/v1' 로 끝나는지 확인하세요.")
        return {"ok": False, "models": [], "error": f"HTTP {response.status_code}"}

    models = [m.get("id", "?") for m in response.json().get("data", [])]

    print(f"\n[성공] 서버가 응답했습니다. 사용 가능한 모델 {len(models)}개:")
    for name in models:
        mark = "  <- 기본 모델" if name == conf["model"] else ""
        print(f"   - {name}{mark}")

    if conf["model"] not in models and models:
        print(
            f"\n[주의] 설정된 모델 '{conf['model']}' 이 목록에 없습니다.\n"
            f"       위 목록에서 골라 LLM(model='...') 로 지정하거나\n"
            f"       환경변수 KMU_MODEL 을 설정하세요."
        )

    return {"ok": True, "models": models, "error": None}


def _mask_url(url: str) -> str:
    """화면 공유·스크린샷 유출을 막기 위해 호스트를 가린다.

    EN — Mask the host so screen shares and screenshots do not leak it.
    """
    import re

    return re.sub(r"//([^/]+)", lambda m: "//" + m.group(1)[:4] + "***", url, count=1)


def _print_troubleshooting() -> None:
    """연결 실패 시 확인할 것들. / What to check when the connection fails."""
    print("\n확인해 볼 것:")
    print("   1. Colab Secrets 의 KMU_BASE_URL 값이 정확합니까?")
    print("      - http / https 를 맞게 적었습니까?")
    print("      - 포트가 필요할 수 있습니다.")
    print("      - 끝이 '/v1' 이어야 합니다. ('/chat/completions' 는 빼도 됩니다)")
    print("   2. 학과 서버가 켜져 있습니까? 교내망에서만 열려 있지 않습니까?")
    print("   3. 다른 주소를 시험해 보려면:")
    print("      check_server(base_url='...')")
    print("   4. 계속 안 되면 백업 경로를 쓰세요:  LLM(provider='gemini')")


In [ ]:
%%writefile memory.py
"""전산실험2 4주차 : 대화 상태와 메모리.

모델은 이전 대화를 기억하지 못한다. 기억은 **우리 코드가** 만든다.
그런데 대화를 통째로 다시 보내다 보면 컨텍스트 윈도우에 부딪힌다.

이 모듈은 세 가지를 제공한다.

    Conversation   대화 기록을 들고 있는 그릇 (system 은 항상 보존)
    예산 관리       토큰을 세고, 넘치면 오래된 턴을 잘라낸다
    요약 압축       잘라내는 대신 요약으로 갈아끼운다 (정보 손실 최소화)

EN — Week 4: conversation state and memory.
The model remembers nothing; OUR code creates the memory. But resending the
whole conversation eventually hits the context window. This module offers:
Conversation (a container for the history, with the system message always
preserved), budget management (count tokens and drop old turns when over), and
summary compression (replace old turns with a summary instead of dropping them).

사용 예 · Example:
    from kmu_llm import LLM
    from memory import Conversation

    llm = LLM()
    conv = Conversation(llm, system="너는 통계 조교다.", max_tokens=2000)
    print(conv.say("표준편차가 뭔가요?"))
    print(conv.say("그럼 표준오차는요?"))     # 앞 대화를 기억한다 / remembers
    conv.summary()                          # 현재 토큰 사용 현황 / token usage
"""

from __future__ import annotations

from typing import Any, Literal

__all__ = ["estimate_tokens", "Message", "Conversation"]

Role = Literal["system", "user", "assistant"]


# --------------------------------------------------------------------------
# 토큰 추정 / Token estimation
# --------------------------------------------------------------------------
def estimate_tokens(text: str) -> int:
    """토큰 수를 대략 추정한다. 서버에 묻지 않고 빠르게 계산한다.

    정확한 값은 모델의 토크나이저에 따라 다르다. 여기서는 경험적 근사를 쓴다.

        한글 1글자  ~ 1.2 토큰   (자소 단위로 잘게 쪼개짐)
        영문 1글자  ~ 0.3 토큰   (단어가 통째로 1토큰이 되는 경우가 많음)

    루프를 돌 때마다 서버에 물어보면 호출이 배로 늘어난다. 예산 관리는
    **근사로 충분**하고, 정확한 값이 필요하면 `LLM.count_tokens()` 를 쓴다.

    EN — Roughly estimate the token count locally, without asking the server.
    The exact number depends on the model's tokenizer; these are empirical
    approximations (one Hangul character ≈ 1.2 tokens because it splits into
    jamo; one Latin character ≈ 0.3 tokens because whole words often become one
    token). Asking the server on every loop iteration would double the number of
    calls — an approximation is good enough for budgeting, and `LLM.count_tokens()`
    is there when you need the exact figure.
    """
    if not text:
        return 0
    hangul = sum(1 for ch in text if "가" <= ch <= "힣")
    other = len(text) - hangul
    return int(hangul * 1.2 + other * 0.3) + 1


class Message(dict):
    """{"role": ..., "content": ...} 에 토큰 수를 얹은 것.

    EN — A {"role": ..., "content": ...} dict with a token count attached.
    """

    def __init__(self, role: Role, content: str) -> None:
        super().__init__(role=role, content=content)

    @property
    def tokens(self) -> int:
        return estimate_tokens(self["content"]) + 4  # 역할 표시 등 부대 비용 / overhead for the role marker

    def as_api(self) -> dict[str, str]:
        return {"role": self["role"], "content": self["content"]}


# --------------------------------------------------------------------------
# 대화 / Conversation
# --------------------------------------------------------------------------
class Conversation:
    """대화 기록을 관리하고 예산을 넘지 않게 유지한다.

    EN — Keeps the conversation history and holds it within the token budget.

    Parameters
    ----------
    llm : LLM 인스턴스 / an LLM instance
    system : 대화 전체에 적용되는 지시. **절대 잘려나가지 않는다.**
             EN — the instruction for the whole conversation; never truncated.
    max_tokens : 전체 대화의 토큰 예산 (응답 공간은 별도)
                 EN — token budget for the conversation (reply space is separate)
    strategy :
        "drop"    오래된 턴을 그냥 버린다 (빠르고 단순, 정보 손실)
                  EN — discard old turns (fast and simple, loses information)
        "summary" 오래된 턴을 요약으로 갈아끼운다 (호출 1회 추가, 손실 적음)
                  EN — replace old turns with a summary (one extra call, less loss)
        "none"    아무것도 하지 않는다 (한계를 직접 보고 싶을 때)
                  EN — do nothing, so you can watch the limit being hit
    keep_recent : 최근 몇 개의 메시지는 무조건 원본으로 보존할지
                  EN — how many recent messages are always kept verbatim
    """

    def __init__(
        self,
        llm: Any,
        system: str | None = None,
        *,
        max_tokens: int = 3000,
        strategy: str = "summary",
        keep_recent: int = 4,
    ) -> None:
        if strategy not in ("drop", "summary", "none"):
            raise ValueError(f"알 수 없는 strategy: {strategy!r}")

        self.llm = llm
        self.max_tokens = max_tokens
        self.strategy = strategy
        self.keep_recent = keep_recent

        self.system = Message("system", system) if system else None
        self.history: list[Message] = []
        self.compressions = 0          # 몇 번 압축했는지 / how many compressions so far
        self.dropped = 0               # 몇 개를 버렸는지 / how many messages were dropped

    # -- 상태 확인 / Inspecting the state ----------------------------------
    def messages(self) -> list[dict[str, str]]:
        """API 로 보낼 형태. / The shape that goes to the API."""
        head = [self.system] if self.system else []
        return [m.as_api() for m in head + self.history]

    @property
    def tokens(self) -> int:
        head = self.system.tokens if self.system else 0
        return head + sum(m.tokens for m in self.history)

    def summary(self) -> dict[str, Any]:
        """현재 상태를 한눈에. / The current state at a glance."""
        return {
            "메시지수": len(self.history) + (1 if self.system else 0),
            "추정토큰": self.tokens,
            "예산": self.max_tokens,
            "사용률": round(self.tokens / self.max_tokens, 3),
            "압축횟수": self.compressions,
            "버린메시지": self.dropped,
        }

    # -- 대화 / Talking ----------------------------------------------------
    def add(self, role: Role, content: str) -> None:
        self.history.append(Message(role, content))

    def say(self, text: str, **kw: Any) -> str:
        """사용자 발화를 넣고 모델의 답을 받아 기록에 남긴다.

        예산은 **보내기 전과 받은 뒤 두 번** 맞춘다. 그래야
        "대화는 항상 예산 안에 있다"는 불변식이 유지되어,
        지금 상태를 보고 다음 호출이 들어갈지 예측할 수 있다.

        EN — Append the user's utterance, get the model's reply, and record it.
        The budget is enforced twice — before sending and after receiving — so
        the invariant "the conversation always fits the budget" holds, which
        lets you predict from the current state whether the next call will fit.
        """
        self.add("user", text)
        self.fit()                                  # 보내기 전 / before sending
        reply = self.llm.chat(self.messages(), **kw)
        self.add("assistant", reply)
        self.fit()                                  # 받은 뒤 / after receiving
        return reply

    # -- 예산 관리 / Budget management -------------------------------------
    def fit(self) -> None:
        """예산을 넘으면 전략에 따라 줄인다.

        EN — Shrink the history according to the strategy when over budget.
        """
        if self.strategy == "none" or self.tokens <= self.max_tokens:
            return
        if self.strategy == "drop":
            self._drop_old()
        else:
            self._summarize_old()

    def _split(self) -> tuple[list[Message], list[Message]]:
        """(압축 대상인 오래된 것, 보존할 최근 것)

        EN — (the old part to compress, the recent part to keep)
        """
        if len(self.history) <= self.keep_recent:
            return [], self.history
        return self.history[: -self.keep_recent], self.history[-self.keep_recent :]

    def _drop_old(self) -> None:
        """가장 오래된 것부터 예산에 맞을 때까지 버린다.

        `keep_recent` 는 보존하려 하지만, 그것만으로도 예산을 넘으면
        보존 약속을 깨고 더 버린다. 마지막 메시지 하나는 남긴다 —
        그것이 사용자의 방금 질문이기 때문이다.

        EN — Drop from the oldest until the history fits. `keep_recent` is
        honoured where possible, but if even those exceed the budget the promise
        is broken and more is dropped. The last message always survives — it is
        the user's current question.
        """
        old, recent = self._split()
        while old and self.tokens > self.max_tokens:
            old.pop(0)
            self.dropped += 1
            self.history = old + recent
        self.history = old + recent

        # 최근 것만 남았는데도 넘치면, 보존 약속을 깨고 더 버린다
        # EN — if even the recent messages overflow, break the promise and drop more
        while len(self.history) > 1 and self.tokens > self.max_tokens:
            self.history.pop(0)
            self.dropped += 1

        if self.tokens > self.max_tokens:
            # 마지막 메시지 하나가 예산보다 크다. 내용을 몰래 자르지는 않는다.
            # 조용히 틀리느니 시끄럽게 알리는 편이 낫다. (3주차와 같은 원칙)
            # EN — a single message exceeds the budget. We do not silently
            #      truncate it: better to fail loudly than to be quietly wrong
            #      (the same principle as week 3).
            print(
                f"[경고] 마지막 메시지 하나가 예산({self.max_tokens} 토큰)을 "
                f"넘습니다 (현재 {self.tokens}). 예산을 늘리거나 입력을 줄이세요."
            )

    def _summarize_old(self) -> None:
        """오래된 대화를 요약 한 덩어리로 갈아끼운다.

        EN — Replace the old part of the conversation with a single summary.
        """
        old, recent = self._split()
        if not old:
            self._drop_old()          # 줄일 것이 최근 것뿐이면 버리는 수밖에 없다
                                      # EN — if only recent turns remain, dropping is the only option
            return

        기록 = "\n".join(f"[{m['role']}] {m['content']}" for m in old)
        지시 = (
            "다음은 사용자와 조수의 지난 대화다. 이후 대화를 이어가는 데 필요한 "
            "사실만 5문장 이내로 요약하라. 결정된 내용, 사용자가 밝힌 정보, "
            "아직 해결되지 않은 질문을 우선 남겨라. 인사말은 버려라.\n\n"
            f"{기록}"
        )
        try:
            요약 = self.llm.ask(지시, temperature=0.0, max_tokens=400)
        except Exception:
            self._drop_old()          # 요약 호출이 실패하면 버리기로 대체
                                      # EN — if the summary call fails, fall back to dropping
            return

        self.history = [Message("user", f"[이전 대화 요약]\n{요약}")] + recent
        self.compressions += 1

        if self.tokens > self.max_tokens:   # 요약해도 넘치면 결국 버린다 / still over: drop
            self._drop_old()

    # -- 저장 / 복원 · Saving and restoring --------------------------------
    def to_dict(self) -> dict[str, Any]:
        return {
            "system": self.system["content"] if self.system else None,
            "history": [m.as_api() for m in self.history],
            "compressions": self.compressions,
            "dropped": self.dropped,
        }

    @classmethod
    def from_dict(cls, llm: Any, data: dict[str, Any], **kw: Any) -> "Conversation":
        conv = cls(llm, system=data.get("system"), **kw)
        for m in data.get("history", []):
            conv.add(m["role"], m["content"])
        conv.compressions = data.get("compressions", 0)
        conv.dropped = data.get("dropped", 0)
        return conv


In [ ]:
!pip install -q openai

# 이 수업 저장소에서 공용 모듈 내려받기
# EN — Load the shared module written by the setup cell above.

from kmu_llm import LLM, ask, check_server

PROVIDER = "kmu"          # 학과 서버가 안 되면 "gemini" 로 바꾸세요 / switch to "gemini" if the department server is down
llm = LLM(provider=PROVIDER)
print("준비 완료:", llm)

---
## 1. 검증 가능한 사실을 심어 둔다 · Planting verifiable facts

긴 분석 상담 대화를 만듭니다. 그 안에 **나중에 확인할 수 있는 사실 8개**를
흩뿌려 둡니다. 압축한 뒤 이것들을 물어볼 것입니다.

> **EN** — We build a long consulting conversation and scatter **eight facts we
> can check later** through it. After compression, we ask about each one.

In [ ]:
사실들 = [
    ("표본크기",  "제 데이터는 환자 137명입니다.",              "137"),
    ("설계",     "치료 전후로 각각 측정한 대응표본입니다.",       "대응"),
    ("변수",     "주 결과변수는 수축기 혈압(mmHg)입니다.",       "수축기"),
    ("유의수준",  "유의수준은 0.01로 엄격하게 잡겠습니다.",       "0.01"),
    ("결측",     "결측치는 12건이고 완전제거법으로 처리했습니다.", "12"),
    ("층화",     "성별로 층화해서 분석하려고 합니다.",           "성별"),
    ("소프트웨어", "최종 보고는 R로 재현할 예정입니다.",          "R"),
    ("기한",     "학회 마감이 3월 20일입니다.",                 "20"),
]

# 사실 사이에 끼워 넣을 평범한 질문들 / ordinary questions to interleave between the facts
일반질문 = [
    "대응표본 t검정의 가정은 무엇인가요?",
    "정규성은 어떻게 확인하나요?",
    "Shapiro-Wilk 검정의 한계는요?",
    "비모수 대안은 무엇이 있나요?",
    "효과크기는 어떤 지표를 쓰나요?",
    "Cohen's d의 해석 기준은요?",
    "신뢰구간도 함께 보고해야 하나요?",
    "다중비교 보정이 필요할까요?",
]

print(f"심어둘 사실 {len(사실들)}개, 사이사이 일반 질문 {len(일반질문)}개")

In [ ]:
from memory import Conversation

SYSTEM = "너는 통계 상담 조교다. 3문장 이내로 간결하게 답한다."

def 대화만들기(strategy, max_tokens=1000, keep_recent=4):
    # EN — Build the same 16-turn conversation under a given strategy.
    conv = Conversation(llm, system=SYSTEM, max_tokens=max_tokens,
                        strategy=strategy, keep_recent=keep_recent)
    # 사실과 일반질문을 번갈아 넣는다 / alternate a planted fact and an ordinary question
    for (_, 사실문장, _), 질문 in zip(사실들, 일반질문):
        conv.say(사실문장, max_tokens=120)
        conv.say(질문, max_tokens=150)
    return conv

print("세 가지 전략으로 같은 대화를 만듭니다 (각 16턴, 몇 분 걸립니다)")
대화들 = {}
for s in ["none", "drop", "summary"]:
    print(f"  {s} ...", end="", flush=True)
    대화들[s] = 대화만들기(s)
    print(f" 완료  {대화들[s].summary()}")

---
## 2. probe — 심어둔 사실을 물어본다 · Probing for the planted facts

압축 후에도 각 사실을 기억하는지 하나씩 확인합니다.
채점은 **정답 키워드가 응답에 들어 있는가**로 합니다.

> **EN** — Check one fact at a time to see whether it survived compression. The
> scoring rule is simply **does the expected keyword appear in the answer.**

In [ ]:
import pandas as pd

질문문구 = {
    "표본크기":   "제 표본 크기가 몇 명이라고 했죠? 숫자만 답하세요.",
    "설계":      "제 연구 설계가 대응표본입니까 독립표본입니까? 한 단어로 답하세요.",
    "변수":      "주 결과변수가 무엇이라고 했죠? 짧게 답하세요.",
    "유의수준":   "제가 정한 유의수준은 얼마죠? 숫자만 답하세요.",
    "결측":      "결측치가 몇 건이라고 했죠? 숫자만 답하세요.",
    "층화":      "무엇으로 층화한다고 했죠? 한 단어로 답하세요.",
    "소프트웨어": "최종 보고를 어떤 소프트웨어로 재현한다고 했죠? 한 단어로.",
    "기한":      "학회 마감이 며칠이라고 했죠? 숫자만 답하세요.",
}

결과 = []
for 전략, conv in 대화들.items():
    for 이름, _, 정답 in 사실들:
        # 원본 대화를 훼손하지 않도록 복사본에 물어본다
        # EN — Ask a copy, so the probe does not disturb the original conversation.
        임시 = Conversation.from_dict(llm, conv.to_dict(),
                                     max_tokens=conv.max_tokens,
                                     strategy="none")
        답 = 임시.say(질문문구[이름], max_tokens=60, temperature=0.0)
        맞음 = 정답.lower() in 답.lower()
        결과.append({"전략": 전략, "사실": 이름, "정답": 정답,
                     "응답": 답[:40], "맞음": 맞음})
        print(f"[{전략:7s}] {이름:8s} {'O' if 맞음 else 'X'}  {답[:35]!r}")
    print()

probe = pd.DataFrame(결과)

In [ ]:
# 전략별 정보 보존율 / Information retention by strategy
보존율 = probe.groupby("전략")["맞음"].agg(["sum", "count", "mean"])
보존율.columns = ["맞힌수", "전체", "보존율"]
보존율 = 보존율.reindex(["none", "drop", "summary"])
보존율.round(3)

In [ ]:
# 어떤 사실이 잘 사라지나? / Which facts disappear most readily?
pd.crosstab(probe["사실"], probe["전략"], probe["맞음"], aggfunc="max").reindex(
    [n for n, _, _ in 사실들]
)[["none", "drop", "summary"]]

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
색 = {"none": "#999999", "drop": "#C44E52", "summary": "#4C72B0"}
전략순 = ["none", "drop", "summary"]
ax.bar(전략순, [보존율.loc[s, "보존율"] for s in 전략순],
       color=[색[s] for s in 전략순])
ax.set_ylim(0, 1.1)
ax.set_ylabel("fact recall rate")
ax.set_title(f"Information retention after compression (n={len(사실들)} facts)")
for i, s in enumerate(전략순):
    ax.text(i, 보존율.loc[s, "보존율"] + 0.03,
            f"{보존율.loc[s, '보존율']:.0%}", ha="center")
plt.tight_layout(); plt.show()

print("토큰 사용량")
for s in 전략순:
    v = 대화들[s].summary()
    print(f"  {s:8s} {v['추정토큰']:5d} 토큰  (압축 {v['압축횟수']}회, 버림 {v['버린메시지']}개)")

### 무엇을 확인했나 · What we confirmed

**1. `none` 은 다 기억하지만 예산을 지키지 못합니다.** 실전에서는 쓸 수 없습니다.

**2. `summary` 가 `drop` 보다 낫지만 완벽하지 않습니다.**

**3. 특히 숫자와 고유명사가 잘 사라집니다.**

세 번째가 중요합니다. 통계 분석 세션에서 "표본 크기 137", "유의수준 0.01",
"결측 12건" 같은 값이 요약에서 빠지면 **이후 판단이 전부 틀어집니다.**

> **요약도 모델이 하는 일입니다. 틀릴 수 있습니다.**

> **EN — 1.** `none` remembers everything but breaks the budget; it is unusable
> in practice.
>
> **EN — 2.** `summary` beats `drop`, but it is not perfect.
>
> **EN — 3.** Numbers and proper nouns are the first to go — and that is the one
> that matters. In a statistics session, losing "137 subjects", "alpha = 0.01" or
> "12 missing values" from the summary **throws off every judgement that follows.**
>
> **Summarising is also something a model does. It can get it wrong.**

---
## 3. 요약 프롬프트 개선하기 · Improving the summarisation prompt

`memory.py` 의 기본 요약 지시는 이렇습니다.

```
이후 대화를 이어가는 데 필요한 사실만 5문장 이내로 요약하라.
결정된 내용, 사용자가 밝힌 정보, 아직 해결되지 않은 질문을 우선 남겨라.
```

숫자 보존을 더 강하게 요구하면 나아질까요? **측정해 봅시다.**

> **EN** — That is the default instruction in `memory.py`: summarise in five
> sentences or fewer, keeping decisions made, information the user gave, and
> questions still open. Would demanding numbers be preserved help? **Let's
> measure.**

In [ ]:
import memory

원본_summarize = memory.Conversation._summarize_old

def 개선된_요약지시(기록):
    # EN — An improved summarisation instruction that demands numbers be kept.
    return (
        "다음은 사용자와 통계 조교의 대화다. 이후 상담을 이어가는 데 필요한 정보를 요약하라.\n"
        "\n"
        "반드시 지킬 것:\n"
        "- 사용자가 말한 **모든 숫자**(표본 크기, 유의수준, 결측 건수, 날짜 등)를 그대로 옮겨 적어라.\n"
        "- 변수명, 소프트웨어명, 고유명사를 그대로 옮겨 적어라.\n"
        "- 연구 설계(대응/독립, 층화 여부)를 명시하라.\n"
        "- 조교의 일반적인 설명은 버려도 좋다.\n"
        "\n"
        "형식: '- 항목: 값' 의 목록으로 쓴다. 인사말은 쓰지 않는다.\n\n"
        f"{기록}"
    )

def 개선된_summarize(self):
    # EN — Monkey-patched _summarize_old that uses the improved instruction.
    old, recent = self._split()
    if not old:
        self._drop_old(); return
    기록 = "\n".join(f"[{m['role']}] {m['content']}" for m in old)
    try:
        요약 = self.llm.ask(개선된_요약지시(기록), temperature=0.0, max_tokens=500)
    except Exception:
        self._drop_old(); return
    self.history = [memory.Message("user", f"[이전 대화 요약]\n{요약}")] + recent
    self.compressions += 1
    if self.tokens > self.max_tokens:
        self._drop_old()

memory.Conversation._summarize_old = 개선된_summarize
print("요약 지시를 교체했습니다.")

In [ ]:
print("개선된 요약으로 다시 대화 생성 중 ...", end="", flush=True)
개선대화 = 대화만들기("summary")
print(" 완료")
print(개선대화.summary())
print()

개선결과 = []
for 이름, _, 정답 in 사실들:
    임시 = Conversation.from_dict(llm, 개선대화.to_dict(),
                                 max_tokens=개선대화.max_tokens, strategy="none")
    답 = 임시.say(질문문구[이름], max_tokens=60, temperature=0.0)
    맞음 = 정답.lower() in 답.lower()
    개선결과.append({"사실": 이름, "맞음": 맞음, "응답": 답[:40]})
    print(f"{이름:8s} {'O' if 맞음 else 'X'}  {답[:35]!r}")

개선 = pd.DataFrame(개선결과)
print()
print(f"기존 summary 보존율 : {보존율.loc['summary', '보존율']:.0%}")
print(f"개선 summary 보존율 : {개선['맞음'].mean():.0%}")

In [ ]:
# 실제로 어떤 요약이 만들어졌는지 봅시다 / Let's see the summary it actually produced
for m in 개선대화.messages():
    if "[이전 대화 요약]" in m["content"]:
        print(m["content"][:900])
        break

> **프롬프트 한 줄이 정보 보존율을 바꿉니다.**
> 3주차에서 배운 것과 같은 교훈입니다. 프롬프트는 코드입니다.
>
> 다만 n=8 로는 개선을 확신할 수 없습니다.
> 연습문제 2에서 반복 측정으로 확인하십시오.

> **EN — one line of prompt changes the retention rate.** The same lesson as week
> 3: a prompt is code. But n = 8 cannot establish the improvement — confirm it by
> repeated measurement in Exercise 2.

---
## 4. 언제 무엇을 쓸 것인가 · When to use which

| 상황 | 권장 | 이유 |
|---|---|---|
| 짧은 단발 질의 | `none` | 압축이 필요 없다 |
| ReAct 루프 안쪽 (6주차) | `drop` | 속도가 중요. 중간 시도는 버려도 된다 |
| 긴 분석 상담 세션 | `summary` | 사용자가 말한 조건을 잃으면 안 된다 |
| 중요 값이 반드시 필요 | **별도 저장** | 요약에 맡기지 말고 구조화해서 따로 들고 있어라 |

**EN**

| Situation | Recommended | Why |
|---|---|---|
| A short one-off query | `none` | No compression needed |
| Inside a ReAct loop (week 6) | `drop` | Speed matters; intermediate attempts are disposable |
| A long consulting session | `summary` | The conditions the user stated must not be lost |
| A value that absolutely must survive | **Store it separately** | Do not entrust it to a summary — hold it in structured form |

**마지막 줄이 가장 중요합니다.**

표본 크기나 유의수준처럼 반드시 지켜야 하는 값이라면
대화 요약에 맡기지 마십시오. **딕셔너리에 따로 담아 두고, 필요할 때
프롬프트에 다시 넣으십시오.** 9주차의 `DataProfiler` 가 그렇게 동작합니다.

> **EN — the last row matters most.** For a value that has to hold, such as the
> sample size or the significance level, do not entrust it to a conversation
> summary. **Keep it in a dictionary and re-inject it into the prompt when
> needed.** That is exactly how the week-9 `DataProfiler` works.

In [ ]:
# 중요한 값은 대화가 아니라 구조화된 상태로 들고 있는다
# EN — Hold the values that matter as structured state, not inside the conversation.
분석설정 = {
    "n": 137,
    "design": "paired",
    "outcome": "systolic_bp_mmHg",
    "alpha": 0.01,
    "missing": 12,
    "stratify_by": "sex",
}

def 설정문구(cfg):
    return "[현재 분석 설정]\n" + "\n".join(f"- {k}: {v}" for k, v in cfg.items())

# 압축이 아무리 일어나도 이 값들은 매 호출에 다시 들어간다
# EN — However often compression fires, these values re-enter on every call.
conv = Conversation(llm, system=SYSTEM + "\n\n" + 설정문구(분석설정),
                    max_tokens=800, strategy="summary")
for q in 일반질문[:6]:
    conv.say(q, max_tokens=120)

print(conv.say("제 표본 크기와 유의수준을 다시 말해 주세요.", max_tokens=100))
print()
print("-> system 에 넣었으므로 절대 사라지지 않습니다.")

---
## 연습문제 · Exercises

**문제 1.** `keep_recent` 를 2, 4, 8 로 바꿔 가며 `summary` 의 보존율을 재시오.
최근 것을 많이 남길수록 좋은가? 토큰 비용과 함께 판단하시오.

**문제 2.** 기존 요약 지시와 개선된 지시를 각각 **5회씩 반복**해 보존율을 재고,
차이를 검정하시오. (사실 8개 x 5회 = 40 시행/조건)

**문제 3.** 요약을 **두 번 이상** 거친 정보는 더 많이 사라지는가?
대화를 32턴으로 늘려 압축이 여러 번 일어나게 한 뒤 보존율을 재시오.
(전화 놀이 효과)

**문제 4.** `분석설정` 처럼 구조화해서 `system` 에 넣는 방식의 단점은 무엇인가?
어떤 정보가 이 방식에 맞고 어떤 정보가 맞지 않는가?

> **EN — 1.** Measure `summary`'s retention with `keep_recent` at 2, 4 and 8. Is
> keeping more recent turns always better? Judge it alongside the token cost.
>
> **EN — 2.** Measure retention for the original and improved instructions with
> **five repetitions each** (8 facts × 5 runs = 40 trials per condition) and test
> the difference.
>
> **EN — 3.** Does information that has passed through **two or more** summaries
> degrade further? Extend the conversation to 32 turns so compression fires
> repeatedly, then measure retention. (The Chinese-whispers effect.)
>
> **EN — 4.** What is the downside of putting structured state into `system`, as
> `분석설정` does? Which kinds of information suit this approach and which do not?

In [ ]:
# 여기에 직접 작성해 보세요 / Write your own code here

<details>
<summary><b>해설 보기 · Show the solutions</b></summary>

**문제 2.** 예시
```python
from statsmodels.stats.proportion import proportions_ztest
성공 = [기존맞힌수, 개선맞힌수]; n = [40, 40]
print(proportions_ztest(성공, n))
```

**문제 3.** 대개 더 사라집니다. 요약의 요약을 만들면 원문에서 멀어지기 때문입니다.
사람의 전화 놀이와 같습니다.

대책: 요약을 **누적**하지 말고, 원본에서 매번 새로 만들거나(비용 큼),
중요한 값을 문제 4처럼 따로 들고 있는 것입니다.

**문제 4.** 단점은 두 가지입니다.

첫째, `system` 이 길어지면 **매 호출마다** 그 토큰을 냅니다. 대화가 20턴이면
20번 다시 보냅니다. 넣을 것은 정말 필요한 것만 넣어야 합니다.

둘째, **구조화할 수 있는 정보만** 가능합니다. "사용자가 결측 처리 방침을
고민하다가 완전제거법으로 기울었지만 확신은 없다" 같은 뉘앙스는 딕셔너리에
안 들어갑니다.

정리하면 **값은 구조화해서 들고, 맥락은 요약에 맡깁니다.**
9주차의 `DataProfiler` 가 정확히 이 분업을 합니다.

---

**EN — 3.** It usually degrades further: summarising a summary drifts away from
the original, exactly like the game of Chinese whispers. The remedy is not to
**accumulate** summaries — either regenerate from the original each time (costly)
or hold the important values separately, as in Exercise 4.

**EN — 4.** Two downsides. First, a longer `system` costs those tokens **on every
call**: a 20-turn conversation resends it 20 times, so put in only what is truly
needed. Second, it only works for **information that can be structured** — a
nuance like "the user wavered on the missing-data policy and leans toward
complete-case deletion, but is not sure" does not fit in a dictionary.

In short: **hold the values as structure, and leave the context to the summary.**
The week-9 `DataProfiler` implements exactly that division of labour.

</details>

---
## 4주차 정리 · Week 4 summary

1. 기억은 모델이 아니라 **우리 코드**가 만든다
2. 대화를 매번 다시 보내므로 누적 토큰은 **제곱으로** 늘어난다
3. 예산 관리는 **근사로 충분**하다. 여유분을 둔다
4. `system` 과 마지막 질문은 **절대 버리지 않는다**
5. 요약도 모델이 한다. **숫자와 고유명사가 사라진다**
6. **값은 구조화해서 따로 들고, 맥락만 요약에 맡긴다**

> **EN**
> 1. Memory is created by **our code**, not by the model
> 2. Because the conversation is resent every turn, cumulative tokens grow
>    **quadratically**
> 3. **An approximation suffices** for budgeting — leave slack
> 4. **Never discard** `system` or the last question
> 5. Summarising is also done by a model — **numbers and proper nouns vanish**
> 6. **Hold values as structure; leave only context to the summary**

## 내 저장소에 저장하기 · Saving it to your repository

`memory.py` 은(는) 이 노트북 위쪽 **준비 셀이 이미 만들어 두었습니다.**
따로 내려받을 것이 없습니다. 자기 저장소로 옮기려면 clone 한 폴더에 복사하세요.

The file(s) above were already written by the setup cells near the top of this
notebook — nothing to download. To keep them in your own repository, copy them
into your cloned folder.

```bash
# repo = 자기 저장소를 clone 한 폴더 / your cloned repository folder
!cp memory.py repo/src/
```


## 다음 주 예고 · Coming next week

지금까지 우리 에이전트는 **말만** 할 수 있습니다.
계산도 못 하고 데이터도 못 봅니다.

다음 주에는 **도구(Tool)** 를 붙입니다. 기술통계를 계산하는 함수,
검정을 수행하는 함수, 그림을 그리는 함수를 만들고 모델이 골라 쓰게 합니다.

1주차에 "LLM에게 난수를 뽑게 하지 마라"고 했던 이유가 그때 분명해집니다.
**계산은 모델이 아니라 도구가 합니다.**

> **EN** — So far our agent can only **talk**: it cannot compute and it cannot see
> data. Next week we attach **tools** — functions that compute descriptive
> statistics, run tests and draw figures — and let the model choose among them.
> The reason behind week 1's "never make an LLM draw random numbers" becomes plain
> then: **the tool computes, not the model.**